# Imports

In [29]:
import os
import json
import time
import itertools
import random

from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from IPython.display import display

# Hyper Parameters
all parameters stay true to eli's paper

In [ ]:
# ============================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================

WEIGHT_DECAYS = [0.0] + np.geomspace(1e-4, 0.2, num=6).tolist()     # logarithmic scale
HIDDEN_WIDTHS = [2, 4, 8, 12, 16, 20, 32, 64, 128, 256, 512]
LABEL_NOISE_PERCENTAGES = [0, 5, 15, 25]  # Percentages


# ============================================================
# TRAINING PARAMETERS
# ============================================================

EPOCHS = 10000
BATCH_SIZE = 512
LEARNING_RATE = 1e-3

OPTIMIZER = "AdamW"

# Dataset
MODULUS = 97
TRAIN_FRACTION = 0.70

# Random seeds
DATA_SEED = 42
MODEL_SEED = 42


# ============================================================
# EVALUATION & CHECKPOINTS
# ============================================================

EVAL_EVERY = 50
CHECKPOINT_EVERY = 1000

TRAIN_ACCURACY_THRESHOLD = 0.99
GROKKING_MIN_DELAY = 200

# Save final trained models
SAVE_MODELS = True

# None = all models
# Example: [0, 1, 2] = only train those indices
RUN_INDICES = None


# ============================================================
# DIRECTORIES
# ============================================================

OUT_DIR = Path("grokking_experiments")
MODELS_DIR = OUT_DIR / "models"
DATASET_DIR = OUT_DIR / "datasets"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_FILE = OUT_DIR / "configurations.json"

# ============================================================
# MODEL TRAINING VISUALIZATION
# ============================================================

SHOW_PLOTS = True
SAVE_PLOTS = True

# ============================================================
# HARDWARE
# ============================================================

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"    # cuda0 since im using ROCM with amd gpu, to prevent confusion
)
print(f"Device: {DEVICE}")

Device: cuda:0


# Pre-defined training essentials

##### Configuration.json - json used to store all model possible configurations(combinations), it's used by the code to then grab a different model for training

##### Datasets - generates |LABEL_NOISE| ammount of datasets labeled as datset_X.csv where X is in LABEL_NOISE and is the % of noise

In [36]:
# ============================================================
# Configuration.json
# ============================================================

# create all possible combinations
configurations = []

for index, (wd, width, noise) in enumerate(
    itertools.product(
        WEIGHT_DECAYS,
        HIDDEN_WIDTHS,
        LABEL_NOISE_PERCENTAGES
    )
):
    configurations.append({
        "index": index,
        "weightDecay": float(wd),
        "hiddenWidth": int(width),
        "labelNoise": int(noise)
    })

# save to json
with open(CONFIG_FILE, "w") as f:
    json.dump(configurations, f, indent=4)

# ============================================================
# datasets
# ============================================================

# Generate all possible input pairs
X = []
y = []

for a in range(MODULUS):
    for b in range(MODULUS):
        X.append([a, b])
        y.append((a + b) % MODULUS)

X = np.array(X)
y = np.array(y)

# train/test split
rng = np.random.default_rng(DATA_SEED)

random_indices_list = rng.permutation(len(X))
train_size = int(len(X) * TRAIN_FRACTION)

train_indices = random_indices_list[:train_size]
test_indices = random_indices_list[train_size:]

X_train = X[train_indices]
y_train_clean = y[train_indices]

X_test = X[test_indices]
y_test = y[test_indices]

# generate and save the datasets
noise_rng = np.random.default_rng(DATA_SEED + 1)
noise_order = noise_rng.permutation(train_size)
noise_offsets = noise_rng.integers(1, MODULUS, size=train_size)

for noise in LABEL_NOISE_PERCENTAGES:

    y_train = y_train_clean.copy()
    n_corrupt = round(train_size * noise / 100)
    corrupt_indices = noise_order[:n_corrupt]

    y_train[corrupt_indices] = (y_train[corrupt_indices] + noise_offsets[corrupt_indices]) % MODULUS

    # Combine training and testing data
    dataset_df = pd.DataFrame({
        "a": np.concatenate((X_train[:, 0], X_test[:, 0])),
        "b": np.concatenate((X_train[:, 1], X_test[:, 1])),
        "label": np.concatenate((y_train, y_test)),
        "cleanLabel": np.concatenate((y_train_clean, y_test)),
        "split": ["train"] * train_size + ["test"] * len(X_test)
    })

    dataset_path = DATASET_DIR / f"dataset_{noise}.csv"
    dataset_df.to_csv(dataset_path, index=False)

    print(f"Saved: {dataset_path.name} | Noise: {noise}% | Corrupted: {n_corrupt}/{train_size}")

print(f"\nsaved {len(LABEL_NOISE_PERCENTAGES)} datasets")


# Load datasets into pytorch
print(f"\nloading datasets into gpu...(hopefully)...")
DATASETS = {}

for noise in LABEL_NOISE_PERCENTAGES:

    # Load CSV
    dataset_path = DATASET_DIR / f"dataset_{noise}.csv"
    df = pd.read_csv(dataset_path)

    # Separate training and testing data
    train_df = df[df["split"] == "train"]
    test_df = df[df["split"] == "test"]

    # Convert directly to PyTorch tensors on GPU
    DATASETS[noise] = {

        "X_train": torch.tensor(
            train_df[["a", "b"]].to_numpy(),
            dtype=torch.long,
            device=DEVICE
        ),

        "y_train": torch.tensor(
            train_df["label"].to_numpy(),
            dtype=torch.long,
            device=DEVICE
        ),

        "y_train_clean": torch.tensor(
            train_df["cleanLabel"].to_numpy(),
            dtype=torch.long,
            device=DEVICE
        ),

        "X_test": torch.tensor(
            test_df[["a", "b"]].to_numpy(),
            dtype=torch.long,
            device=DEVICE
        ),

        "y_test": torch.tensor(
            test_df["cleanLabel"].to_numpy(),
            dtype=torch.long,
            device=DEVICE
        )
    }

    print(f"Loaded: Dataset {noise:2d}% | Train: {len(train_df):,} | Test: {len(test_df):,} | Device: {DEVICE}")

Saved: dataset_0.csv | Noise: 0% | Corrupted: 0/6586
Saved: dataset_5.csv | Noise: 5% | Corrupted: 329/6586
Saved: dataset_15.csv | Noise: 15% | Corrupted: 988/6586
Saved: dataset_25.csv | Noise: 25% | Corrupted: 1646/6586

saved 4 datasets

loading datasets into gpu...(hopefully)...
Loaded: Dataset  0% | Train: 6,586 | Test: 2,823 | Device: cuda:0
Loaded: Dataset  5% | Train: 6,586 | Test: 2,823 | Device: cuda:0
Loaded: Dataset 15% | Train: 6,586 | Test: 2,823 | Device: cuda:0
Loaded: Dataset 25% | Train: 6,586 | Test: 2,823 | Device: cuda:0


# Model Architecture

input(2) ->

-> shared embedding(MODULUS * hidden_dim) ->

-> embedding(2 * hidden_dim) ->

-> concat(2 * hidden_dim) ->

-> hidden1(2 * hidden_dim) -> Relu ->

-> hidden2(hidden_dim) -> Relu ->

-> Output(97) -> ArgMax 

In [37]:
class GrokkingMLP(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(MODULUS, hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * hidden_dim, 4 * hidden_dim),
            nn.ReLU(),
            nn.Linear(4 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, MODULUS)
        )

    def forward(self, x):

        # x shape (batch_size, 2)
        x = self.embedding(x)

        # Shape: (batch_size, 2*d)
        x = x.flatten(start_dim=1)

        # Output shape: (batch_size, 97)
        return self.mlp(x)

# Evaluation

In [38]:
@torch.inference_mode()
def evaluate_model(model, dataset, batch_size=BATCH_SIZE):

    current_model_mode = model.training
    model.eval()

    X_train = dataset["X_train"]                
    y_train = dataset["y_train"]
    y_train_clean = dataset["y_train_clean"]

    X_test = dataset["X_test"]
    y_test = dataset["y_test"]


    # training metrics
    total_train_loss = 0.0
    correct_train = 0
    correct_clean_train = 0

    for start in range(0, len(X_train), batch_size):

        end = start + batch_size

        outputs = model(X_train[start:end])

        # Cross-entropy loss against noisy training labels
        loss = F.cross_entropy(outputs, y_train[start:end],eduction="sum")

        total_train_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct_train += (predictions == y_train[start:end]).sum().item()

        correct_clean_train += (predictions == y_train_clean[start:end]).sum().item()

    train_loss = total_train_loss / len(X_train)
    train_accuracy = correct_train / len(X_train)
    clean_train_accuracy = (correct_clean_train / len(X_train))


    # Test metrics
    correct_test = 0

    for start in range(0, len(X_test), batch_size):

        end = start + batch_size

        outputs = model(X_test[start:end])

        predictions = outputs.argmax(dim=1)

        correct_test += (
            predictions == y_test[start:end]
        ).sum().item()

    test_accuracy = correct_test / len(X_test)


    # Restore original model mode
    model.train(current_model_mode)

    
    return {
        "trainLoss": train_loss,
        "trainAccuracy": train_accuracy,
        "cleanTrainAccuracy": clean_train_accuracy,
        "testAccuracy": test_accuracy
    }

# Grokking detection

#### criteria:  
1. rain convergence: Accuracy ≈ 100%.  
2. Validation convergence: Accuracy ≥ 1 − noise (e.g., 85%).  
3. Minimum delay: Δt= t_val − t_train ≥ T₁.
4. Time limit: t_val ≤ T₂.
5. Color: Δt(green = faster, red = slower).
6. Missing point: No grokking detected by T₂.


In [44]:
def detect_grokking(
    history,
    label_noise,
    train_threshold=TRAIN_ACCURACY_THRESHOLD,
    min_delay=GROKKING_MIN_DELAY,
    max_epoch=EPOCHS
):

    df = pd.DataFrame(history).copy()

    required_columns = ["epoch", "trainAccuracy", "testAccuracy"]

    if not all(col in df.columns for col in required_columns):
        raise ValueError("Training history is missing required columns.")

    # Sort by epoch and ignore measurements beyond T2
    df = df.sort_values("epoch")
    df = df[df["epoch"] <= max_epoch]
    df = df.drop_duplicates("epoch", keep="last")

    # ========================================================
    # 1. TRAINING CONVERGENCE: Accuracy ≈ 100%
    # 2. VALIDATION CONVERGENCE: Accuracy >= 1 - noise
    # ========================================================

    test_threshold = 1 - (label_noise / 100)

    def first_convergence_epoch(column, threshold):

        converged = df[df[column] >= threshold]

        if converged.empty:
            return None

        return int(converged.iloc[0]["epoch"])

    t_train = first_convergence_epoch(
        "trainAccuracy",
        train_threshold
    )

    t_val = first_convergence_epoch(
        "testAccuracy",
        test_threshold
    )

    # ========================================================
    # 3. MINIMUM DELAY: delta_t >= T1
    # ========================================================

    delay = None

    if t_train is not None and t_val is not None:
        delay = t_val - t_train

    # ========================================================
    # 4. TIME LIMIT: t_val <= T2
    # ========================================================

    within_time_limit = (
        t_val is not None
        and t_val <= max_epoch
    )

    # ========================================================
    # 5-6. GROKKING CLASSIFICATION
    # ========================================================

    grokked = (
        t_train is not None
        and within_time_limit
        and delay >= min_delay
    )

    # ========================================================
    # RETURN RESULTS
    # ========================================================

    return {
        "trainConvergenceEpoch": t_train,
        "testConvergenceEpoch": t_val,
        "grokkingDelay": delay,
        "grokked": bool(grokked),
        "trainThreshold": train_threshold,
        "testThreshold": test_threshold
    }

# Model training function

In [ ]:
def train_model(config):

    # --------------------------------------------------------
    # 1. read model config
    # --------------------------------------------------------

    index = int(config["index"])
    weight_decay = float(config["weightDecay"])
    hidden_width = int(config["hiddenWidth"])
    label_noise = int(config["labelNoise"])

    # create dir for the model
    model_dir = MODELS_DIR / f"model_{index:03d}"
    model_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = model_dir / "checkpoint.pt"
    model_path = model_dir / "model.pt"
    history_path = model_dir / "history.csv"
    summary_path = model_dir / "summary.json"

    # skip completed models
    if summary_path.exists():
        with open(summary_path, "r") as f:
            summary = json.load(f)

        if any(summary.get(k) != v for k, v in config.items()):
            raise ValueError(
                f"Model {index}: existing results use a different configuration."
            )

        print(f"Model {index}: Already completed. Skipping.")
        return summary

    # --------------------------------------------------------
    # 2. select the correct dataset
    # --------------------------------------------------------

    dataset = DATASETS[label_noise]

    X_train = dataset["X_train"]
    y_train = dataset["y_train"]

    # --------------------------------------------------------
    # 3. model initialization
    # --------------------------------------------------------

    torch.manual_seed(MODEL_SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_SEED)

    model = GrokkingMLP(hidden_dim=hidden_width).to(DEVICE)

    # --------------------------------------------------------
    # 4. optimizer and loss functions
    # --------------------------------------------------------

    if OPTIMIZER.lower() != "adamw":
        raise ValueError(f"Unsupported optimizer: {OPTIMIZER}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=weight_decay
    )

    loss_fn = nn.CrossEntropyLoss()

    # --------------------------------------------------------
    # 5. training state
    # --------------------------------------------------------

    start_epoch = 1
    history = []

    elapsed_seconds = 0.0

    # --------------------------------------------------------
    # 6. resume from checkpoint if found
    # --------------------------------------------------------

    if checkpoint_path.exists():

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False
        )

        if checkpoint["config"] != config:
            raise ValueError(
                f"Model {index}: checkpoint configuration mismatch."
            )

        model.load_state_dict(checkpoint["model_state_dict"])

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"] + 1

        history = checkpoint["history"]

        elapsed_seconds = checkpoint.get(
            "elapsedSeconds", 0.0
        )

        print(
            f"Model {index}: Resuming from epoch {start_epoch}"
        )

    # --------------------------------------------------------
    # 7. actual training loop
    # --------------------------------------------------------

    print(f"\nTraining Model {index:03d} | Width={hidden_width} | WD={weight_decay} | Noise={label_noise}%")

    start_time = time.time()

    for epoch in range(start_epoch, EPOCHS + 1):

        model.train()

        # shuffle training data
        generator = torch.Generator().manual_seed(MODEL_SEED + epoch)

        shuffled_indices = torch.randperm(len(X_train),generator=generator).to(DEVICE)

        # batch training
        for batch_indices in shuffled_indices.split(BATCH_SIZE):

            optimizer.zero_grad(set_to_none=True)

            outputs = model(X_train[batch_indices]) # forward propagation

            loss = loss_fn(outputs,y_train[batch_indices])  # calculate loss

            loss.backward() # back propagation

            optimizer.step()    # update weights

        # ----------------------------------------------------
        # 8. eval
        # ----------------------------------------------------

        if (epoch == 1 or epoch % EVAL_EVERY == 0 or epoch == EPOCHS):

            metrics = evaluate_model(model,dataset)

            history.append({"epoch": epoch,**metrics})

        # ----------------------------------------------------
        # 9. checkpoint every X epochs (incase it crashes or something)
        # ----------------------------------------------------

        if epoch % CHECKPOINT_EVERY == 0:

            checkpoint = {
                "epoch": epoch,
                "config": config,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "history": history,
                "elapsedSeconds": (
                    elapsed_seconds
                    + time.time() - start_time
                )
            }

            # prevents checkpoint from replacing a valid one
            temp_path = model_dir / "checkpoint.tmp"

            torch.save(checkpoint, temp_path)

            temp_path.replace(checkpoint_path)

        # ----------------------------------------------------
        # 10. display progress
        # ----------------------------------------------------

        if epoch % 500 == 0 or epoch == EPOCHS:

            latest = history[-1]

            print(f"Model {index:03d} | Epoch {epoch:5d}/{EPOCHS} | Train: {latest['trainAccuracy']:.4f} | Test: {latest['testAccuracy']:.4f}")

    # --------------------------------------------------------
    # 11. detect grokking(hopefully)
    # --------------------------------------------------------

    grokking = detect_grokking(history=history,label_noise=label_noise)

    # --------------------------------------------------------
    # 12. save results
    # --------------------------------------------------------

    if SAVE_MODELS:

        torch.save(model.state_dict(),model_path)

    pd.DataFrame(history).to_csv(history_path,index=False)

    summary = {
        **config,
        "epochs": EPOCHS,
        "parameters": count_parameters(model),
        **grokking,
        "finalTrainAccuracy": history[-1]["trainAccuracy"],
        "finalCleanTrainAccuracy": history[-1]["cleanTrainAccuracy"],
        "finalTestAccuracy": history[-1]["testAccuracy"],
        "elapsedSeconds": (
            elapsed_seconds
            + time.time() - start_time
        )
    }

    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=4)

    # The final model has been saved successfully.
    # Remove the temporary resume checkpoint.
    checkpoint_path.unlink(missing_ok=True)

    print(f"\nModel {index:03d} completed! | Grokked: {grokking['grokked']} | Test accuracy: {summary['finalTestAccuracy']:.4f}")

    return summary

# Visualization

In [ ]:
def plot_model_results(model_index, show=SHOW_PLOTS):

    model_dir = MODELS_DIR / f"model_{model_index:03d}"

    # Load saved results
    history = pd.read_csv(model_dir / "history.csv")

    with open(model_dir / "summary.json", "r") as f:
        summary = json.load(f)

    # Model configuration
    width = summary["hiddenWidth"]
    wd = summary["weightDecay"]
    noise = summary["labelNoise"]

    # Grokking detection results
    t_train = summary["trainConvergenceEpoch"]
    t_val = summary["testConvergenceEpoch"]
    delay = summary["grokkingDelay"]
    grokked = summary["grokked"]

    # Convergence thresholds
    train_threshold = summary["trainThreshold"]
    test_threshold = summary["testThreshold"]

    # ========================================================
    # CREATE FIGURE
    # ========================================================

    fig, axes = plt.subplots(
        1, 3,
        figsize=(19, 5),
        constrained_layout=True
    )

    status = "Grokking Detected" if grokked else "No Grokking"
    delay_text = f"{delay} epochs" if delay is not None else "N/A"

    fig.suptitle(
        f"Model {model_index:03d} | "
        f"Width={width} | WD={wd:.6g} | Noise={noise}%\n"
        f"{status} | Delay: {delay_text}",
        fontsize=14,
        fontweight="bold"
    )

    # ========================================================
    # 1-2. ACCURACY CURVES
    # ========================================================

    for ax in axes[:2]:

        ax.plot(
            history["epoch"],
            history["trainAccuracy"],
            label="Train (Noisy)",
            linewidth=2
        )

        ax.plot(
            history["epoch"],
            history["cleanTrainAccuracy"],
            label="Train (Clean)",
            linewidth=2
        )

        ax.plot(
            history["epoch"],
            history["testAccuracy"],
            label="Test",
            linewidth=2
        )

        # Training convergence threshold
        ax.axhline(
            train_threshold,
            color="gray",
            linestyle="--",
            alpha=0.6,
            label=f"Train Threshold ({train_threshold:.0%})"
        )

        # Validation convergence threshold
        ax.axhline(
            test_threshold,
            color="green",
            linestyle="--",
            alpha=0.6,
            label=f"Test Threshold ({test_threshold:.0%})"
        )

        # Training convergence epoch
        if t_train is not None:
            ax.axvline(
                t_train,
                color="blue",
                linestyle=":",
                alpha=0.8,
                label=f"Train Convergence ({t_train})"
            )

        # Validation convergence epoch
        if t_val is not None:
            ax.axvline(
                t_val,
                color="red",
                linestyle=":",
                alpha=0.8,
                label=f"Test Convergence ({t_val})"
            )

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Accuracy")
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    # ========================================================
    # 1. FULL TRAINING HISTORY
    # ========================================================

    axes[0].set_title("Full Training History")

    # ========================================================
    # 2. ZOOM AROUND VALIDATION CONVERGENCE
    # ========================================================

    ax = axes[1]

    max_epoch = history["epoch"].max()

    if t_val is not None:

        # Show 500 epochs before and after convergence
        zoom_start = max(0, t_val - 500)
        zoom_end = min(max_epoch, t_val + 500)

        ax.set_title("Validation Convergence (Zoom)")

    else:

        # If validation never converged, show early training
        zoom_start = 0
        zoom_end = min(1500, max_epoch)

        ax.set_title("Early Training (Zoom)")

    ax.set_xlim(zoom_start, max(zoom_end, zoom_start + 1))

    # ========================================================
    # 3. TRAINING LOSS
    # ========================================================

    ax = axes[2]

    ax.plot(
        history["epoch"],
        history["trainLoss"],
        linewidth=2,
        label="Training Loss"
    )

    ax.set_title("Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()

    # ========================================================
    # SAVE AND DISPLAY
    # ========================================================

    if SAVE_PLOTS:

        figure_path = model_dir / "learning_curves.png"

        fig.savefig(
            figure_path,
            dpi=150,
            bbox_inches="tight"
        )

        print(f"Saved plot: {figure_path}")

    if show:
        plt.show()
    else:
        plt.close(fig)

# Actual model training runs

In [ ]:
# ============================================================
# AUTOMATED EXPERIMENT RUNNER
# ============================================================

from tqdm.auto import tqdm


# ============================================================
# 1. LOAD AND VALIDATE CONFIGURATIONS
# ============================================================

with open(CONFIG_FILE, "r") as f:
    experiment_configs = json.load(f)

config_by_index = {
    config["index"]: config
    for config in experiment_configs
}

# Ensure configuration indices are unique
if len(config_by_index) != len(experiment_configs):
    raise ValueError("Duplicate configuration indices detected!")

print(f"Total configurations: {len(experiment_configs)}")


# ============================================================
# 2. SELECT CONFIGURATIONS TO TRAIN
# ============================================================

if RUN_INDICES is None:

    selected_configs = experiment_configs

else:

    requested_indices = set(RUN_INDICES)

    missing_indices = requested_indices - set(config_by_index)

    if missing_indices:
        raise ValueError(
            f"Unknown configuration indices: {sorted(missing_indices)}"
        )

    selected_configs = [
        config
        for config in experiment_configs
        if config["index"] in requested_indices
    ]


print(f"Selected configurations: {len(selected_configs)}")
print(f"Epochs per model: {EPOCHS:,}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Checkpoint interval: {CHECKPOINT_EVERY}")
print(f"Device: {DEVICE}")


# ============================================================
# 3. VALIDATE EXISTING EXPERIMENTS
# ============================================================

def get_experiment_status(config):

    index = config["index"]

    model_dir = MODELS_DIR / f"model_{index:03d}"

    summary_path = model_dir / "summary.json"
    history_path = model_dir / "history.csv"
    checkpoint_path = model_dir / "checkpoint.pt"

    # --------------------------------------------------------
    # COMPLETED EXPERIMENT
    # --------------------------------------------------------

    if summary_path.exists():

        with open(summary_path, "r") as f:
            summary = json.load(f)

        # Validate configuration
        for key, value in config.items():

            if summary.get(key) != value:
                raise ValueError(
                    f"Model {index:03d}: configuration mismatch "
                    f"for '{key}'."
                )

        # Validate training duration
        if summary.get("epochs") != EPOCHS:
            raise ValueError(
                f"Model {index:03d}: saved for "
                f"{summary.get('epochs')} epochs, "
                f"but current EPOCHS={EPOCHS}."
            )

        # Ensure training history exists
        if not history_path.exists():
            raise FileNotFoundError(
                f"Model {index:03d}: summary exists "
                f"but history.csv is missing."
            )

        return "completed"

    # --------------------------------------------------------
    # RESUMABLE EXPERIMENT
    # --------------------------------------------------------

    if checkpoint_path.exists():

        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False
        )

        # Validate checkpoint configuration
        if checkpoint.get("config") != config:
            raise ValueError(
                f"Model {index:03d}: checkpoint "
                f"configuration mismatch."
            )

        # Validate checkpoint epoch
        checkpoint_epoch = checkpoint.get("epoch", -1)

        if not 1 <= checkpoint_epoch <= EPOCHS:
            raise ValueError(
                f"Model {index:03d}: invalid checkpoint "
                f"epoch {checkpoint_epoch}."
            )

        return "resumable"

    # --------------------------------------------------------
    # NEW EXPERIMENT
    # --------------------------------------------------------

    return "pending"


# Check selected experiments
status_counts = {
    "completed": 0,
    "resumable": 0,
    "pending": 0
}

for config in selected_configs:

    status = get_experiment_status(config)

    status_counts[status] += 1


print("\n" + "=" * 60)
print("EXPERIMENT STATUS")
print("=" * 60)

print(f"Completed:   {status_counts['completed']}")
print(f"Resumable:   {status_counts['resumable']}")
print(f"Not started: {status_counts['pending']}")


# ============================================================
# 4. COMBINED RESULTS SAVING
# ============================================================

RESULTS_FILE = OUT_DIR / "results.csv"


def save_experiment_results():

    summaries = []

    # Only collect results belonging to the current experiment
    for config in experiment_configs:

        index = config["index"]

        summary_path = (
            MODELS_DIR
            / f"model_{index:03d}"
            / "summary.json"
        )

        if not summary_path.exists():
            continue

        with open(summary_path, "r") as f:
            summary = json.load(f)

        # Validate configuration
        if any(
            summary.get(key) != value
            for key, value in config.items()
        ):
            raise ValueError(
                f"Model {index:03d}: saved configuration mismatch."
            )

        # Validate epoch count
        if summary.get("epochs") != EPOCHS:
            raise ValueError(
                f"Model {index:03d}: saved epoch count mismatch."
            )

        summaries.append(summary)

    if not summaries:
        return pd.DataFrame()

    # Combine model summaries
    results_df = pd.DataFrame(summaries)

    results_df = (
        results_df
        .sort_values("index")
        .reset_index(drop=True)
    )

    # Save atomically to avoid corrupting an existing results file
    temp_path = RESULTS_FILE.with_suffix(".tmp")

    results_df.to_csv(
        temp_path,
        index=False
    )

    temp_path.replace(RESULTS_FILE)

    return results_df


# ============================================================
# 5. RUN EXPERIMENTS
# ============================================================

print("\nStarting automated experiments...\n")

experiment_start = time.time()

interrupted = False

try:

    for config in tqdm(
        selected_configs,
        desc="Training Models",
        unit="model"
    ):

        index = config["index"]

        # ----------------------------------------------------
        # TRAIN OR RESUME MODEL
        # ----------------------------------------------------

        train_model(config)

        # ----------------------------------------------------
        # SAVE COMBINED RESULTS
        # ----------------------------------------------------

        save_experiment_results()

        # ----------------------------------------------------
        # GENERATE MODEL VISUALIZATION
        # ----------------------------------------------------

        plot_model_results(
            model_index=index,
            show=SHOW_PLOTS
        )

except KeyboardInterrupt:

    interrupted = True

    print("\nTraining interrupted by user.")
    print("Completed models and saved checkpoints are preserved.")

finally:

    # Save all completed results, even after interruption
    results_df = save_experiment_results()


# ============================================================
# 6. FINAL EXPERIMENT SUMMARY
# ============================================================

elapsed_seconds = time.time() - experiment_start

print("\n" + "=" * 60)
print("EXPERIMENT SUMMARY")
print("=" * 60)

if interrupted:
    print("Status: INTERRUPTED")
else:
    print("Status: FINISHED")

print(f"Execution time: {elapsed_seconds / 3600:.2f} hours")

print(f"Total configurations: {len(experiment_configs)}")

print(f"Completed models: {len(results_df)}")

print(
    f"Remaining models: "
    f"{len(experiment_configs) - len(results_df)}"
)

print(f"Results file: {RESULTS_FILE}")


# ============================================================
# 7. DISPLAY RESULTS
# ============================================================

if not results_df.empty:

    grokked_count = int(results_df["grokked"].sum())

    grokking_rate = (
        grokked_count / len(results_df) * 100
    )

    print(f"\nModels exhibiting Grokking: {grokked_count}")
    print(f"Grokking detection rate: {grokking_rate:.1f}%")

    columns = [
        "index",
        "weightDecay",
        "hiddenWidth",
        "labelNoise",
        "trainConvergenceEpoch",
        "testConvergenceEpoch",
        "grokkingDelay",
        "grokked",
        "finalTestAccuracy"
    ]

    print("\nFirst 20 completed experiments:")

    display(
        results_df[columns].head(20)
    )

else:

    print("\nNo experiments completed yet.")

# Grokking visualization

In [ ]:
# ============================================================
# GROKKING LANDSCAPE VISUALIZATION
# ============================================================

from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


def plot_grokking_landscape(show=True, save=True):

    # ========================================================
    # 1. LOAD EXPERIMENT DATA
    # ========================================================

    results_path = OUT_DIR / "results.csv"

    if not results_path.exists():
        raise FileNotFoundError(
            f"{results_path} not found. "
            "Complete at least one model first."
        )

    with open(CONFIG_FILE, "r") as f:
        configurations = json.load(f)

    results = pd.read_csv(results_path)

    required_columns = {
        "index",
        "hiddenWidth",
        "weightDecay",
        "labelNoise",
        "grokked",
        "grokkingDelay"
    }

    missing = required_columns - set(results.columns)

    if missing:
        raise ValueError(
            f"Missing results columns: {sorted(missing)}"
        )

    # ========================================================
    # 2. VALIDATE RESULTS
    # ========================================================

    configs_by_index = {
        int(config["index"]): config
        for config in configurations
    }

    if len(configs_by_index) != len(configurations):
        raise ValueError(
            "Duplicate indices in configurations.json."
        )

    if results["index"].duplicated().any():
        raise ValueError(
            "Duplicate model indices in results.csv."
        )

    # Prevent mixing results from different configuration grids
    for _, row in results.iterrows():

        index = int(row["index"])

        if index not in configs_by_index:
            raise ValueError(
                f"Model {index} is not in configurations.json."
            )

        config = configs_by_index[index]

        if (
            int(row["hiddenWidth"]) != int(config["hiddenWidth"])
            or int(row["labelNoise"]) != int(config["labelNoise"])
            or not np.isclose(
                float(row["weightDecay"]),
                float(config["weightDecay"]),
                rtol=1e-8,
                atol=1e-12
            )
        ):
            raise ValueError(
                f"Model {index}: configuration mismatch."
            )

    # Convert saved values to appropriate types
    results["grokkedFlag"] = (
        results["grokked"]
        .astype(str)
        .str.lower()
        .isin(["true", "1"])
    )

    results["grokkingDelay"] = pd.to_numeric(
        results["grokkingDelay"],
        errors="coerce"
    )

    if results.loc[
        results["grokkedFlag"], "grokkingDelay"
    ].isna().any():
        raise ValueError(
            "A detected Grokking run is missing its delay."
        )

    # ========================================================
    # 3. PREPARE AXES AND COLOR SCALE
    # ========================================================

    widths = sorted({
        int(config["hiddenWidth"])
        for config in configurations
    })

    weight_decays = sorted({
        float(config["weightDecay"])
        for config in configurations
    })

    noise_levels = sorted({
        int(config["labelNoise"])
        for config in configurations
    })

    completed_indices = set(
        results["index"].astype(int)
    )

    # Use the same color scale across all noise levels
    delays = results.loc[
        results["grokkedFlag"],
        "grokkingDelay"
    ]

    norm = None

    if not delays.empty:

        min_delay = float(delays.min())
        max_delay = float(delays.max())

        if min_delay == max_delay:
            min_delay -= 1
            max_delay += 1

        norm = Normalize(
            vmin=min_delay,
            vmax=max_delay
        )

    # ========================================================
    # 4. CREATE FIGURE
    # ========================================================

    ncols = 2
    nrows = (len(noise_levels) + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(17, 5.5 * nrows),
        sharex=True,
        sharey=True,
        constrained_layout=True
    )

    axes = np.asarray(axes).ravel()

    fig.suptitle(
        "Grokking Landscape — Modular Addition (mod 97)\n"
        "Colored dot = Grokking detected  |  "
        "Blank = No Grokking by T₂  |  "
        "× = Not yet trained",
        fontsize=15,
        fontweight="bold"
    )

    positive_wd = [
        wd for wd in weight_decays
        if wd > 0
    ]

    # ========================================================
    # 5. PLOT EACH NOISE LEVEL
    # ========================================================

    for ax, noise in zip(axes, noise_levels):

        # All configurations expected for this noise level
        expected = [
            config
            for config in configurations
            if int(config["labelNoise"]) == noise
        ]

        # Completed experiments
        subset = results[
            results["labelNoise"] == noise
        ]

        # Only experiments where Grokking was detected
        detected = subset[
            subset["grokkedFlag"]
        ]

        # Experiments that have not completed
        pending = [
            config
            for config in expected
            if int(config["index"]) not in completed_indices
        ]

        # ----------------------------------------------------
        # PLOT DETECTED GROKKING
        # ----------------------------------------------------

        if not detected.empty:

            ax.scatter(
                detected["hiddenWidth"],
                detected["weightDecay"],
                c=detected["grokkingDelay"],
                cmap="RdYlGn_r",
                norm=norm,
                s=165,
                edgecolors="black",
                linewidths=0.35,
                zorder=3
            )

        # ----------------------------------------------------
        # MARK UNFINISHED EXPERIMENTS
        # ----------------------------------------------------

        if pending:

            ax.scatter(
                [config["hiddenWidth"] for config in pending],
                [config["weightDecay"] for config in pending],
                marker="x",
                color="0.6",
                s=60,
                linewidths=1.1,
                zorder=2
            )

        # ----------------------------------------------------
        # GRAPH TITLE
        # ----------------------------------------------------

        ax.set_title(
            f"Label Noise: {noise}%  |  "
            f"Grokking: {len(detected)}/{len(subset)} completed",
            fontsize=12
        )

        # ----------------------------------------------------
        # X-AXIS: NETWORK WIDTH
        # ----------------------------------------------------

        ax.set_xscale("log", base=2)

        ax.set_xticks(widths)

        ax.set_xticklabels(
            [str(width) for width in widths],
            rotation=40,
            fontsize=9
        )

        ax.tick_params(
            axis="x",
            labelbottom=True
        )

        ax.set_xlim(
            min(widths) / 1.25,
            max(widths) * 1.25
        )

        # ----------------------------------------------------
        # Y-AXIS: WEIGHT DECAY
        # ----------------------------------------------------

        # Symmetric log supports WD=0
        if positive_wd:

            min_positive = min(positive_wd)

            ax.set_yscale(
                "symlog",
                base=10,
                linthresh=min_positive / 10
            )

            ax.set_ylim(
                -min_positive / 8,
                max(weight_decays) * 1.4
            )

        else:
            ax.set_ylim(-0.1, 0.1)

        ax.set_yticks(weight_decays)

        ax.set_yticklabels([
            f"{wd:.3g}"
            for wd in weight_decays
        ])

        # ----------------------------------------------------
        # LABELS AND GRID
        # ----------------------------------------------------

        ax.set_xlabel("Network Width (d)")
        ax.set_ylabel("Weight Decay (log scale, with zero)")

        ax.grid(
            True,
            which="major",
            linestyle=":",
            alpha=0.3
        )

    # Hide unused panels if necessary
    for ax in axes[len(noise_levels):]:
        ax.set_visible(False)

    # ========================================================
    # 6. SHARED COLORBAR
    # ========================================================

    if norm is not None:

        colorbar = fig.colorbar(
            ScalarMappable(
                norm=norm,
                cmap="RdYlGn_r"
            ),
            ax=axes[:len(noise_levels)].tolist(),
            shrink=0.82,
            pad=0.02
        )

        colorbar.set_label(
            "Grokking Delay Δt (epochs)\n"
            "Green = faster, Red = slower"
        )

    # ========================================================
    # 7. SAVE FIGURE
    # ========================================================

    if save:

        plot_path = OUT_DIR / "grokking_landscape.png"

        fig.savefig(
            plot_path,
            dpi=170,
            bbox_inches="tight"
        )

        print(f"Saved: {plot_path}")

    # ========================================================
    # 8. DISPLAY SUMMARY
    # ========================================================

    grokked_count = int(
        results["grokkedFlag"].sum()
    )

    pending_count = (
        len(configurations) - len(results)
    )

    print(
        f"Grokking detected: "
        f"{grokked_count}/{len(results)} completed models"
    )

    print(f"Not yet completed: {pending_count}")

    if show:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# GENERATE GROKKING LANDSCAPE
# ============================================================

plot_grokking_landscape()